# USDA NASS QuickStats — Yields, Production, Acreage & Prices

**What it is.** The official U.S. agricultural statistics database — virtually every
survey/census estimate NASS publishes: **yield, production, harvested/planted acres,
price received, stocks, and the Census of Agriculture**.

| | |
|---|---|
| **Coverage** | United States — national, state, **county**, and ag-district |
| **History** | Survey data back to ~1866; Census of Ag every 5 years |
| **Resolution** | County-level (finest routinely available) |
| **Cost / key** | Free · **API key required** (you already have `NASS_API_KEY`) |
| **Docs** | https://quickstats.nass.usda.gov/api |

**Why it's core to Tracera.** This is the ground-truth yield & price history a farmer
benchmarks against — "what does an average acre in my county actually produce, and what
did it sell for?"

In [1]:
# Tracera shared setup — credentials + the ONE sample farm location
# (Every source is queried at this same spot so results are comparable.)
import sys, pathlib, requests, pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd()))
from data_sources._common import SAMPLE_FARM, get_key, field_polygon

LAT, LON = SAMPLE_FARM["lat"], SAMPLE_FARM["lon"]
FIPS, STATE, COUNTY = SAMPLE_FARM["fips"], SAMPLE_FARM["state_alpha"], SAMPLE_FARM["county_name"]
print(SAMPLE_FARM["name"], f"| lat={LAT}, lon={LON} | FIPS {FIPS}")

Story County, Iowa (sample farm) | lat=42.05, lon=-93.5 | FIPS 19169


### 1. Connection test — most recent corn yield for the sample county

In [2]:
BASE = "https://quickstats.nass.usda.gov/api/api_GET/"

def quickstats(**params):
    """Query NASS QuickStats and return a tidy DataFrame."""
    params = {"key": get_key("NASS_API_KEY"), "format": "JSON", **params}
    r = requests.get(BASE, params=params, timeout=60)
    if r.status_code == 400:              # NASS returns 400 when a query is too broad/empty
        raise ValueError(r.json().get("error", r.text))
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

test = quickstats(commodity_desc="CORN", statisticcat_desc="YIELD",
                  agg_level_desc="COUNTY", state_alpha=STATE, county_name=COUNTY,
                  year__GE="2020")
assert not test.empty, "No rows returned"
print(f"OK — {len(test)} rows")
test[["year", "short_desc", "Value", "unit_desc"]].head()

OK — 7 rows


,year,short_desc,Value,unit_desc
0,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",204,BU / ACRE
1,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",215.5,BU / ACRE
2,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",211.4,BU / ACRE
3,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",207.1,BU / ACRE
4,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",201.2,BU / ACRE


### 2. Core query — county corn & soybean **yield** history (bu/acre)

In [3]:
frames = []
for crop in ("CORN", "SOYBEANS"):
    df = quickstats(commodity_desc=crop, statisticcat_desc="YIELD",
                    agg_level_desc="COUNTY", state_alpha=STATE, county_name=COUNTY,
                    year__GE="2000", reference_period_desc="YEAR")
    frames.append(df)
yields = pd.concat(frames, ignore_index=True)
yields["Value"] = pd.to_numeric(yields["Value"], errors="coerce")
yields["year"] = pd.to_numeric(yields["year"], errors="coerce")
pivot = (yields.pivot_table(index="year", columns="commodity_desc", values="Value")
               .sort_index())
print("Yield (bu/acre) — tail:")
pivot.tail(8)

Yield (bu/acre) — tail:


commodity_desc,CORN,SOYBEANS
year,,
2018,192.30,52.8
2019,189.30,50.0
2020,148.40,52.2
2021,201.20,63.6
2022,207.10,60.0
2023,211.40,64.1
2024,119.75,63.2
2025,204.00,63.6


### 3. Core query — **price received** for corn ($/bu), monthly

In [4]:
prices = quickstats(commodity_desc="CORN", statisticcat_desc="PRICE RECEIVED",
                    state_alpha=STATE, year__GE="2020",
                    short_desc="CORN, GRAIN - PRICE RECEIVED, MEASURED IN $ / BU")
prices["Value"] = pd.to_numeric(prices["Value"], errors="coerce")
prices = prices[prices["reference_period_desc"] != "MARKETING YEAR"]
print("Latest corn prices ($/bu):")
prices[["year", "reference_period_desc", "Value"]].head(6)

Latest corn prices ($/bu):


,year,reference_period_desc,Value
0,2026,JAN,4.13
1,2026,FEB,4.12
2,2026,MAR,4.28
3,2026,APR,4.29
4,2026,MAY,4.45
6,2025,JAN,4.37


### Notes & how to extend
- **Key filters:** `commodity_desc`, `statisticcat_desc` (YIELD / PRODUCTION / AREA HARVESTED /
  PRICE RECEIVED), `agg_level_desc` (NATIONAL / STATE / COUNTY / AGRICULTURAL DISTRICT),
  `state_alpha`, `county_name`, `year`, `year__GE`, `year__LE`.
- **Operators:** append `__LE`, `__GE`, `__LT`, `__GT`, `__LIKE` to any field.
- **Gotcha:** a query returning >50,000 rows is rejected — always narrow by commodity + geography.
- **Census of Ag:** add `source_desc="CENSUS"` for operation counts, demographics, practices.
- **Next step:** move `quickstats()` into `data_sources/quickstats.py` for reuse by the app.